# Audio-Dominant Multimodal Emotion Recognition
### Deep Learning-Based Audio-Visual Fusion on eNTERFACE’05

This notebook implements an audio-dominant multimodal fusion framework for emotion classification using speech and facial features.

In [ ]:
import os
import numpy as np
import cv2
import subprocess
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

Dataset Configuration

In [ ]:
ROOT = "path_to_enterface_dataset"
EMOTIONS = ['anger','disgust','fear','happiness','sadness','surprise']
label2idx = {e:i for i,e in enumerate(EMOTIONS)}

AUDIO_NPY = "/content/audio_npy"
VIDEO_NPY = "/content/video_npy"
os.makedirs(AUDIO_NPY, exist_ok=True)
os.makedirs(VIDEO_NPY, exist_ok=True)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def extract_audio(video_path, wav_path):
    if not os.path.exists(wav_path):
        cmd = ["ffmpeg", "-y", "-i", video_path, "-ac", "1", "-ar", "16000", wav_path]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Feature Extraction for video and audio
def preprocess_video_fast(path):
    cap = cv2.VideoCapture(path)
    frames = []
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        count += 1
        if count % 7 != 0:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        if len(faces) > 0:
            x,y,w,h = faces[0]
            face = gray[y:y+h, x:x+w]
        else:
            face = gray

        face = cv2.resize(face, (48, 48))
        frames.append(face)

        if len(frames) == 20:
            break

    cap.release()

    if len(frames) < 20:
        frames.extend([np.zeros((48,48))] * (20 - len(frames)))

    MAX_FRAMES = 16
    frames = frames[:MAX_FRAMES]
    arr = np.stack(frames, axis=0)[..., np.newaxis]

    return arr.astype("float32")


def preprocess_audio_file(path):
    y, sr = librosa.load(path, sr=16000, mono=True)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    feats = np.vstack([mfcc, delta, delta2])

    if feats.shape[1] < 250:
        pad = 250 - feats.shape[1]
        feats = np.pad(feats, ((0,0),(0,pad)), mode='constant')
    else:
        feats = feats[:, :250]

    return feats.T.astype("float32")


#load samples
audio_paths, video_paths, labels = [], [], []

for subject in os.listdir(ROOT):
    sub_path = os.path.join(ROOT, subject)
    if not os.path.isdir(sub_path):
        continue

    for emotion in EMOTIONS:
        emo_path = os.path.join(sub_path, emotion)
        if not os.path.isdir(emo_path):
            continue

        for sentence in os.listdir(emo_path):
            sent_path = os.path.join(emo_path, sentence)
            if not os.path.isdir(sent_path):
                continue

            avi_list = [f for f in os.listdir(sent_path) if f.endswith(".avi")]
            if len(avi_list) == 0:
                continue

            video_path = os.path.join(sent_path, avi_list[0])
            wav_path = os.path.join(sent_path, "audio.wav")

            extract_audio(video_path, wav_path)

            aud = preprocess_audio_file(wav_path)
            vid = preprocess_video_fast(video_path)

            clip_id = f"{subject}_{emotion}_{sentence}"
            aud_npy = os.path.join(AUDIO_NPY, clip_id + ".npy")
            vid_npy = os.path.join(VIDEO_NPY, clip_id + ".npy")

            np.save(aud_npy, aud)
            np.save(vid_npy, vid)

            audio_paths.append(aud_npy)
            video_paths.append(vid_npy)
            labels.append(label2idx[emotion])

print("DONE.")
print("Samples loaded:", len(labels))

X_audio = np.array([np.load(p) for p in audio_paths])
X_video = np.array([np.load(p) for p in video_paths])
y_all  = tf.keras.utils.to_categorical(labels, num_classes=6)

print("X_audio:", X_audio.shape)
print("X_video:", X_video.shape)
print("Labels:", y_all.shape)


DONE.
Samples loaded: 1287
X_audio: (1287, 250, 120)
X_video: (1287, 16, 48, 48, 1)
Labels: (1287, 6)


In [ ]:
# Train/Val Split and class balancing
indices = np.arange(len(y_all))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.15,
    stratify=np.argmax(y_all, axis=1),
    random_state=42
)
X_audio_train = X_audio[train_idx]
X_audio_val   = X_audio[val_idx]

X_video_train = X_video[train_idx]
X_video_val   = X_video[val_idx]

y_train = y_all[train_idx]
y_val   = y_all[val_idx]


In [ ]:

y_all = tf.keras.utils.to_categorical(labels, num_classes=6)

tf.random.set_seed(42)
np.random.seed(42)

y_int = np.argmax(y_all, axis=1)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(X_audio, y_int))

X_audio_train, X_audio_val = X_audio[train_idx], X_audio[val_idx]
X_video_train = X_video[train_idx][:, :16]
X_video_val   = X_video[val_idx][:, :16]
y_train, y_val = y_all[train_idx], y_all[val_idx]
y_train_int = y_int[train_idx]


class_weights = dict(enumerate(
    compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_int),
        y=y_train_int
    )
))

# Note: Training takes approximately ~1 hour depending on hardware.
#audio model
def build_audio_model():
    inp = layers.Input((250,120))

    x = layers.GaussianNoise(0.02)(inp)  # audio augmentation

    x = layers.Conv1D(
        64, 5, padding='same', activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(
        128, 5, padding='same', activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(2)(x)

    x = layers.Conv1D(
        256, 3, padding='same', activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(0.40)(x)  # 🔹 increased

    emb = layers.Dense(
        64, activation='relu',
        kernel_regularizer=regularizers.l2(1e-4),
        name="audio_embedding"
    )(x)

    out = layers.Dense(6, activation='softmax')(emb)
    return models.Model(inp, out)

audio_model = build_audio_model()
audio_model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

history_audio=audio_model.fit(
    X_audio_train, y_train,
    validation_data=(X_audio_val, y_val),
    epochs=30,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=6, restore_best_weights=True),
        ReduceLROnPlateau(patience=3, factor=0.5)
    ],
    verbose=1
)

#video model
def build_video_model():
    inp = layers.Input((16,48,48,1))

    x = layers.TimeDistributed(
        layers.Conv2D(32,3,padding='same',activation='relu')
    )(inp)
    x = layers.TimeDistributed(layers.MaxPool2D(2))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(64,3,padding='same',activation='relu')
    )(x)
    x = layers.TimeDistributed(layers.MaxPool2D(2))(x)

    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.TimeDistributed(layers.Dense(96,activation='relu'))(x)

    x_avg = layers.GlobalAveragePooling1D()(x)
    x_max = layers.GlobalMaxPooling1D()(x)
    x = layers.Concatenate()([x_avg, x_max])

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    emb = layers.Dense(32, activation='relu', name="video_embedding")(x)
    out = layers.Dense(6, activation='softmax')(emb)

    return models.Model(inp, out)

video_model = build_video_model()
video_model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_video=video_model.fit(
    X_video_train, y_train,
    validation_data=(X_video_val, y_val),
    epochs=25,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=6, restore_best_weights=True),
        ReduceLROnPlateau(patience=3, factor=0.5)
    ],
    verbose=1
)

#embdeddings
audio_embedder = models.Model(
    audio_model.input,
    audio_model.get_layer("audio_embedding").output
)

video_embedder = models.Model(
    video_model.input,
    video_model.get_layer("video_embedding").output
)

A_train = tf.keras.utils.normalize(audio_embedder.predict(X_audio_train), axis=1)
A_val   = tf.keras.utils.normalize(audio_embedder.predict(X_audio_val), axis=1)

V_train = tf.keras.utils.normalize(video_embedder.predict(X_video_train), axis=1)
V_val   = tf.keras.utils.normalize(video_embedder.predict(X_video_val), axis=1)



In [ ]:
#audio dominant fusion

video_projection = layers.Dense(64, use_bias=False, name="video_projection")

V_train_proj = video_projection(V_train).numpy()
V_val_proj   = video_projection(V_val).numpy()

# ↓ reduced interaction strength
F_train = np.concatenate([
    A_train,
    0.3 * V_train_proj,
    0.3 * (A_train * V_train_proj)
], axis=1)

F_val = np.concatenate([
    A_val,
    0.3 * V_val_proj,
    0.3 * (A_val * V_val_proj)
], axis=1)

print("Fused feature shape:", F_train.shape)

#fusion classifier
fusion_model = tf.keras.Sequential([
    layers.Input(shape=(F_train.shape[1],)),
    layers.GaussianNoise(0.05),          
    layers.Dense(
        64, activation='relu',            
        kernel_regularizer=regularizers.l2(2e-4)  
    ),
    layers.BatchNormalization(),
    layers.Dropout(0.60),                
    layers.Dense(6, activation='softmax')
])

fusion_model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4), 
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.10),
    metrics=['accuracy']
)
history_fusion = fusion_model.fit(
    F_train, y_train,
    validation_data=(F_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[
        EarlyStopping(patience=6, restore_best_weights=True),
        ReduceLROnPlateau(patience=3, factor=0.5)
    ],
    verbose=1
)


#evaluation
y_pred = np.argmax(fusion_model.predict(F_val, verbose=0), axis=1)
y_true = np.argmax(y_val, axis=1)

print("\nValidation Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=EMOTIONS))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=EMOTIONS,
    yticklabels=EMOTIONS,
    cmap='Blues'
)
plt.title("Fusion Model – Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
